# A2-Lite — 512-image geometry-only distillation
Latest revision: **A2-LITE-512-GEOMETRY-KD-2026-10-14-r1**
Staging update: **A2-LITE-IO-STAGING-2026-10-14-r1** — four I/O workers, verified partial-file reuse and progress estimates; experiment identity and training code unchanged.

One new experiment: keep teacher guidance for **depth, object size and orientation**, and remove the direct teacher classification and box losses. All ground-truth losses stay unchanged. Class/box information still filters teacher-to-object matches. Geometry can still affect shared features and detection outputs; this is not a freeze of those heads.

Same student: MobileNetV4 Medium, one visual encoder / one detection decoder, 256 channels, 50 queries, 640×192 input. Same teacher: frozen original A2 epoch130. Same exact 512 TRAIN image IDs/order, original untrained pruned weights, seed123, batch1 and five epochs (2,560 updates). This is a 512-image subset, NOT a batch of512 images. The subset balances frame groups (256 pedestrian-containing and256 vehicle-only), not individual object counts.

Compare the new fixed epoch5 endpoint with the two reviewed, already-completed 512-image endpoints: label-only control and all-head distillation. We do NOT retrain these references or use the later full-dataset experiment as a baseline. This single-seed historical comparison is a screening ablation, not a fresh simultaneous repeat or proof that either removed head caused the earlier recall loss.

Teacher terms: log-depth Smooth L1, GT-size-normalized dimension Smooth L1, and orientation-bin KL plus teacher-bin residual Smooth L1. Ground truth: sigmoid focal classification (weight2), 3D projected-center L1 (10), box-extent L1 (5), GIoU (2), uncertainty-weighted depth (1), relative-dimension L1 (1), and angle-bin cross-entropy plus residual L1 (1). KD uses the same class-balanced means, temperature2, maximum weight0.25 ramped over128 updates, correct-class/confidence/2D-IoU matching, and per-component teacher-error≤90% of student-error gate. Optional dense depth-map loss stays off.

Predeclared benefit guardrail versus label-only: all five AP metrics nonregressing, mean moderate3D AP gain≥0.1 point, and nearby recall drop≤2 absolute percentage points for EITHER class. Nearby recall is based on 2D IoU≥0.5 and score≥0.001, Vehicle≤40m / Pedestrian≤30m; it does not mean correct3D reconstruction. Raw changes versus all-head KD are reported too. Product qualification is separate.

Use a fresh GPU Colab runtime (A100 preferred). No phone is needed. No architecture change, teacher inference, native extension build, old virtual environment, Torch/CUDA replacement or Core ML. Ordinary package versions must match the original pilot; if Python/Torch differ, stop for review of fresh matched controls rather than rebuilding an old CUDA stack.

Run sections1–8 in order. After a reset, rerun1–5 and resume6. Checkpoints save every128 updates; up to127 interrupted updates may repeat. Return the ZIP even when a quality check fails. No longer run, checkpoint selection or deployment follows automatically.

## 1. Mount Drive, paths and all helpers
No other notebook cells are required. Edit paths here only if your existing Drive folders differ.


In [ ]:
# A2-LITE-512-GEOMETRY-KD-2026-10-14-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-512-GEOMETRY-KD-2026-10-14-r1'
STAGING_REVISION='A2-LITE-IO-STAGING-2026-10-14-r1'
RUN_ID='a2_lite_512_geometry_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
# ORIGINAL512 pilot folder or original a2_lite_recovery_results.zip, NOT full-recovery results.
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_GEOMETRY_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_geometry_'+RUN_ID)
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_a2_lite_640_accuracy_r1',
    '/content/kitti_m68','/content/kitti_m67','/content/kitti',
    '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_geometry')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_geometry.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True);tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        code=process.wait();log.write(f'\nEXIT CODE: {code}\n');log.flush()
    if code:raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nOutput:',OUTPUT,'\nOne new512-image geometry arm; old references read-only; no phone needed.')


## 2. Update main, ordinary dependencies and runtime check
Section1 above defines every path/helper. PILOT must point to the ORIGINAL512-image run. Torch/CUDA are not replaced. A changed Python/Torch version stops for review; never overwrite the old manifest to force a comparison.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError(f'{SCRIPT}: load the published revision of this notebook')
for path in (PILOT,INITIALIZATION,COMPACT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section1 paths')
manifest_path=OUTPUT/'geometry_manifest.json'
if manifest_path.exists():saved=json.loads(manifest_path.read_text())['cpu_environment']
elif PILOT.is_dir():saved=json.loads((PILOT/'recovery_manifest.json').read_text())['cpu_environment']
else:
    with zipfile.ZipFile(PILOT) as archive:
        matches=[n for n in archive.namelist() if n=='recovery_manifest.json' or n.endswith('/recovery_manifest.json')]
        if len(matches)!=1:raise RuntimeError('Expected one original512 pilot manifest')
        saved=json.loads(archive.read(matches[0]))['cpu_environment']
# Check immutable comparison requirements BEFORE ordinary dependency installation.
probe=("import sys,json,torch; saved=json.loads(sys.argv[1]); "
       "actual={'python':'.'.join(map(str,sys.version_info[:2])),'torch':torch.__version__.split('+')[0]}; "
       "changes={k:{'saved':saved[k],'current':v} for k,v in actual.items() if saved[k]!=v}; "
       "assert not changes,'Stop for fresh matched-reference review; do not restore old CUDA: '+json.dumps(changes)")
run_logged([PYTHON,'-c',probe,json.dumps(saved)],'reference_python_torch')
versions={name:saved[name] for name in ('numpy','timm','onnxruntime','pillow','scipy','opencv-python-headless')}
packages=[name+'=='+version for name,version in versions.items()]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnxruntime as ort;assert torch.cuda.is_available(),'Select a fresh GPU Colab runtime';torch.cuda.init();assert timm.__version__=='1.0.27';assert ort.__version__=='1.30.0';print(torch.__version__,torch.cuda.get_device_name(0),'portable tensor training; no native attention extension')"],'runtime_probe')
READY=True


## 3. Tests and exact historical comparison setup
Verify the original512 pilot reports/cache, original initialization, portable source and all TRAIN/VAL data hashes. Copy/hash all7,481 frames (22,443 files) with four I/O workers. Already verified local files are reused without reading Drive again; progress shows copies, reuse and estimated remaining time. The unchanged experiment prepare then checks local data. GPU idle during I/O is expected. Do not switch this live session to CPU and back: local /content data may be lost when the runtime is replaced. For repeat sessions, a separately prepared archive on Drive is the next input-cache option, not enabled here. No old run is modified and no reference endpoint is retrained.


In [ ]:
if not READY:raise RuntimeError('Run sections1 and2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_data_staging.py','-v'],'staging_tests',MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_geometry_kd.py','-v'],'geometry_tests',MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite*recovery.py','-v'],'frozen_recovery_tests',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
STAGING_SCRIPT=MOBILE_REPO/'scripts/stage_a2_lite_data.py'
if not STAGING_SCRIPT.is_file():raise FileNotFoundError('Pull latest main in section2 before staging')
print(STAGING_REVISION,'— CPU/disk I/O only; training budget unchanged')
run_logged([PYTHON,'-u',STAGING_SCRIPT,'--pilot',PILOT,'--dataset',DATASET,'--workers','4',*extra],'parallel_staging',MOBILE_REPO)
stage('prepare','--pilot',PILOT,'--initialization',INITIALIZATION,'--compact',COMPACT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)
MANIFEST=json.loads((OUTPUT/'geometry_manifest.json').read_text())
print('Exact original TRAIN subset:',len(MANIFEST['pilot_ids']),'fixed updates:',512*5)


## 4. Original640 initialization parity and tiny learnability check
Reproduce all16 CPU fixtures; verify CUDA ground-truth gradients on a disposable copy. Run the same128-update /16-TRAIN-image diagnostic. The diagnostic checkpoint NEVER initializes the512-image arm. A failed quality check skips later training; return section8 results.


In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP: return section8 ZIP; do not lower the gate.')


## 5. Reuse the exact512 TRAIN teacher cache and check geometry gradients
No new teacher inference. Verify every old teacher record, image/calibration/label hash and current encoded-GT signature. Then check depth, size and angle teaching each have finite nonzero parameter gradients, both classes have eligible geometry examples, classification/box teacher counts are zero, and frozen buffers are unchanged. This gradient check performs ZERO optimizer updates.


In [ ]:
KD_READY=False
if TINY_PASSED:
    stage('cache','--pilot',PILOT)
    CACHE=json.loads((OUTPUT/'teacher_cache_complete.json').read_text())
    print(json.dumps({k:CACHE[k] for k in ('images','encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized','teacher_inference_performed')},indent=2))
    if CACHE['kd_pilot_authorized']:
        stage('kd-smoke')
        KD_CHECK=json.loads((OUTPUT/'geometry_kd_smoke.json').read_text());KD_READY=bool(KD_CHECK['passed'])
        print(json.dumps({k:KD_CHECK[k] for k in ('passed','active_counts','component_gradient_abs_sum','optimizer_steps','buffers_unchanged')},indent=2))
    else:print('STOP: teacher cache coverage did not qualify; return section8 evidence.')
else:print('Skipped: tiny learnability check failed.')


## 6. Train ONE new arm — five fixed epochs
2,560 updates, batch1, original pruned weights, same deterministic frame order, allGT losses unchanged. Only direct teacher classification/box terms are absent. KD weight ramps to0.25. Same learning rates, no augmentation/dropout/AMP, frozen BatchNorm/early backbone. Folder kd/ here contains the new geometry-only checkpoint, not the old all-head checkpoint. Resume this section after rerunning1–5; do not change losses or RUN_ID midway.


In [ ]:
if KD_READY:
    stage('train')
else:print('No geometry training started; return the preflight evidence.')


## 7. Evaluate the fixed epoch5 endpoint and show the three-way result
All3,769 VAL frames; no best-epoch selection. Report geometry-only versus label-only and all-head KD, including both nearby-recall deltas. Historical references have the same initialization/subset/budget/software, but are not a fresh simultaneous repeat. Passing relative benefit does not pass product targets or qualify deployment.


In [ ]:
if KD_READY:
    stage('evaluate')
    stage('compare')
    REPORT=json.loads((OUTPUT/'geometry_comparison.json').read_text())
    for name,value in REPORT['results'].items():
        print(name,json.dumps({'metrics':value['metrics'],'nearby_recall':{c:value['nearby']['classes'][c]['near_recall'] for c in ('Vehicle','Pedestrian')}},indent=2))
    print(json.dumps({k:REPORT[k] for k in ('versus_label_only','versus_all_heads','geometry_benefit_guardrail_passed','comparison_limit','full_training_authorized','deployment_qualified')},indent=2))
else:print('Validation skipped: preflight did not qualify training.')


## 8. Download results, also after a failure
Return a2_lite_geometry_results.zip. Includes metrics, predictions, cache identities and combined logs; excludes large weights/ONNX/images. Checkpoints stay on Drive for resume and a separately reviewed export. No phone is needed yet.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_geometry_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Script unavailable; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):
                archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
